[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shuhrat-1/MLP_assignments/blob/main//assignment_2_petrignano/petrignano_assignment_2.ipynb)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import (
    train_test_split, KFold, TimeSeriesSplit,
    GridSearchCV, cross_val_score
)
from sklearn.metrics import r2_score

## Step 0 — Organize data, handle missing values, build X / y

In [ ]:
# --- LOAD ---
df = pd.read_csv('https://raw.githubusercontent.com/Shuhrat-1/MLP_assignments/main/assignment_2_petrignano/Petrignano.csv')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date').set_index('Date')

# --- MISSING VALUES: forward-fill only (no leakage) ---
df = df.ffill()

# --- RESAMPLE: daily → monthly ---
df_monthly = df.resample('ME').agg({
    'Rainfall_Bastia_Umbra':                 'sum',
    'Depth_to_Groundwater_P25':              'mean',
    'Temperature_Bastia_Umbra':              'mean',
    'Temperature_Petrignano':                'mean',
    'Volume_C10_Petrignano':                 'sum',
    'Hydrometry_Fiume_Chiascio_Petrignano':  'mean'
})

# --- BUILD LAGGED FEATURE MATRIX (t-1 and t-2 only, as required) ---
df_lagged = pd.DataFrame(index=df_monthly.index)

for col in df_monthly.columns:
    df_lagged[f"{col}_lag1"] = df_monthly[col].shift(1)   # month t-1
    df_lagged[f"{col}_lag2"] = df_monthly[col].shift(2)   # month t-2

# --- SEASONALITY (optional, as allowed by assignment) ---
df_lagged['month_sin'] = np.sin(2 * np.pi * df_lagged.index.month / 12)
df_lagged['month_cos'] = np.cos(2 * np.pi * df_lagged.index.month / 12)

# --- RESPONSE VARIABLE: current month's groundwater depth ---
df_lagged['Depth_to_Groundwater_P25'] = df_monthly['Depth_to_Groundwater_P25']

# --- DROP NaN rows introduced by lags ---
df_model = df_lagged.dropna()

# --- X and y, sorted chronologically ---
y = df_model['Depth_to_Groundwater_P25']
X = df_model.drop(columns=['Depth_to_Groundwater_P25'])

print(f"Dataset: {X.shape[0]} months × {X.shape[1]} features")
print(f"Period:  {X.index[0].date()} → {X.index[-1].date()}")
X.head(3)

## Step 1 — Hard Chronological Train / Test Split

In [ ]:
# ensure X and y are pre-sorted chronologically!
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print(f"Train: {X_train_full.index[0].date()} → {X_train_full.index[-1].date()}  ({len(X_train_full)} months)")
print(f"Test:  {X_test.index[0].date()} → {X_test.index[-1].date()}   ({len(X_test)} months)")

## Step 2 — Define the Two CV Strategies

In [ ]:
cv_naive    = KFold(n_splits=5, shuffle=True, random_state=42)
cv_temporal = TimeSeriesSplit(n_splits=5)

## Step 3 — Experiment with a Fixed Model

**Expected result:**  
- Naive CV → inflated R² (data leakage: model "sees the future" during shuffled folds)  
- Temporal CV → lower, more honest R² that better predicts actual test performance

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', DecisionTreeRegressor(max_depth=10, random_state=42))
])

# Strategy 1: Naive (Shuffled)
# This will likely report a very high R2 because of leakage.
scores_naive = cross_val_score(
    pipe, X_train_full, y_train_full, cv=cv_naive, scoring='r2'
)

# Strategy 2: Temporal (Sequential)
# This will report a lower, more honest R2.
scores_temporal = cross_val_score(
    pipe, X_train_full, y_train_full, cv=cv_temporal, scoring='r2'
)

print(f"Naive CV R2:    {scores_naive.mean():.4f} (+/- {scores_naive.std():.4f})")
print(f"Temporal CV R2: {scores_temporal.mean():.4f} (+/- {scores_temporal.std():.4f})")

# Train on all training data and test on the unseen "future"
pipe.fit(X_train_full, y_train_full)
final_test_r2 = r2_score(y_test, pipe.predict(X_test))

print(f"\nActual Test R2 on held-out data: {final_test_r2:.4f}")
print()
print("Interpretation:")
print(f"  Naive CV gap  : {scores_naive.mean() - final_test_r2:+.4f}  (optimistic bias from leakage)")
print(f"  Temporal CV gap: {scores_temporal.mean() - final_test_r2:+.4f}  (honest estimate, much closer to reality)")

### Visualise the leakage problem

In [ ]:
labels = ['Naive CV\n(KFold, shuffled)', 'Temporal CV\n(TimeSeriesSplit)', 'Actual Test R²\n(held-out future)']
values = [scores_naive.mean(), scores_temporal.mean(), final_test_r2]
errors = [scores_naive.std(), scores_temporal.std(), 0]
colors = ['#e74c3c', '#3498db', '#2ecc71']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, yerr=errors, color=colors, capsize=8,
              edgecolor='black', linewidth=0.8, width=0.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_ylabel('R² Score', fontsize=12)
ax.set_title('CV Strategy Comparison: Naive vs Temporal vs True Test', fontsize=13)
ax.set_ylim(-0.3, 1.1)
ax.grid(axis='y', alpha=0.3)

fig.text(0.5, -0.03,
    'Red bar (Naive) is inflated due to data leakage — it trains on "future" months.\n'
    'Blue bar (Temporal) honestly reflects out-of-sample performance.',
    ha='center', fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Model Selection & Evaluation

We use **GridSearchCV** with each CV strategy to tune `max_depth` and `min_samples_leaf`.  
The key comparison: how big is the gap between `Internal CV Score` and `Independent Test Score`?  
- **Naive K-Fold** → large gap (CV overly optimistic due to leakage)  
- **Temporal Split** → small gap (CV score is a realistic predictor of test performance)

In [ ]:
def evaluate_model_selection(X_train, y_train, X_test, y_test, cv_strategy, name):

    # STEP A: Define the Pipeline
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', DecisionTreeRegressor(random_state=42))
    ])

    # STEP B: Define the Hyperparameter Grid (4+ depths as required)
    param_grid = {
        'regressor__max_depth':        [2, 3, 5, 7, 10, 12, None],
        'regressor__min_samples_leaf': [1, 2, 4, 6]
    }

    # STEP C: Initialize and Fit GridSearchCV
    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=cv_strategy,
        scoring='r2'
    )
    grid.fit(X_train, y_train)

    # STEP D: Final Evaluation on unseen test set
    y_pred  = grid.predict(X_test)
    test_r2 = r2_score(y_test, y_pred)

    print(f"\n===== Results for: {name} =====")
    print(f"Best Parameters found:       {grid.best_params_}")
    print(f"Internal CV Score (R2):      {grid.best_score_:.4f}")
    print(f"Independent Test Score (R2): {test_r2:.4f}")
    print(f"Gap (CV - Test):             {grid.best_score_ - test_r2:+.4f}")

    return grid, y_pred

## Step 5 — Running the Comparison

In [ ]:
result_naive, y_pred_naive = evaluate_model_selection(
    X_train_full, y_train_full, X_test, y_test,
    cv_naive, "Naive K-Fold"
)

result_temporal, y_pred_temporal = evaluate_model_selection(
    X_train_full, y_train_full, X_test, y_test,
    cv_temporal, "Temporal Split"
)

print()
print("=" * 55)
print("CONCLUSION:")
print("  Naive K-Fold reports inflated CV scores because")
print("  shuffling breaks temporal order → data leakage.")
print("  TimeSeriesSplit CV score is a much better predictor")
print("  of true out-of-sample (future) performance.")
print("=" * 55)

### Predictions on the Test Set

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

for ax, preds, grid, label, color in zip(
    axes,
    [y_pred_naive, y_pred_temporal],
    [result_naive, result_temporal],
    ['Naive K-Fold', 'Temporal Split'],
    ['#e74c3c', '#3498db']
):
    test_r2 = r2_score(y_test, preds)
    ax.plot(y_test.index, y_test.values,  color='black',  linewidth=2,   label='Actual')
    ax.plot(y_test.index, preds,           color=color,    linewidth=1.8,
            linestyle='--', label=f'Predicted (R²={test_r2:.3f})')
    ax.fill_between(y_test.index, y_test.values, preds, alpha=0.15, color=color)
    ax.set_title(f'Test Set Predictions — {label}  |  Best params: {grid.best_params_}', fontsize=11)
    ax.set_ylabel('Depth to GW P25 (m)')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

axes[1].set_xlabel('Date')
plt.suptitle('Naive K-Fold vs Temporal Split — Test Set Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('model_selection_comparison.png', dpi=150, bbox_inches='tight')
plt.show()